In [35]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [36]:
model =ChatOpenAI(model="gpt-4o-mini")

In [37]:
class EvaluationScheama(BaseModel):
    feedback: str = Field(description="detailed feedback for the essay")
    score: int = Field(description="Scour out of 10",ge=0,le=10)
    

In [38]:
structured_model =model.with_structured_output(EvaluationScheama)

In [ ]:
essay = "hello "

In [ ]:
prompt = f'Evaluate the languate quality of the following essay and provide feedback and assign a score out of 10 {essay}'
structured_model.invoke(prompt)

In [50]:
class UpscState(TypedDict):
    essay: str
    language_feedback : str
    analysis_feedback: str
    clarity_feedback : str
    overall_feedback : str
    individual_score : Annotated[list[int],operator.add]
    avg_score: float

In [54]:
def evaluate_language(state: UpscState):
    prompt = f'Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 {state["essay"]}'
    result = structured_model.invoke(prompt)

    return {
        'language_feedback': result.feedback,
        'individual_score': [result.score]
    }

In [51]:
def evaluate_analysis(state: UpscState):
    prompt = f'Evaluate the analysis quality of the following essay and provide feedback and assign a score out of 10 {state["essay"]}'
    result = structured_model.invoke(prompt)

    return {
        'analysis_feedback': result.feedback,
        'individual_score': [result.score]
    }

In [52]:
def evaluate_thought(state: UpscState):
    prompt = f'Evaluate the clarity of thought in the following essay and provide feedback and assign a score out of 10 {state["essay"]}'
    result = structured_model.invoke(prompt)

    return {
        'clarity_feedback': result.feedback,
        'individual_score': [result.score]
    }

In [55]:
def final_evaluation(state:UpscState):
    avg_score = sum(state["individual_score"])/len(state["individual_score"])
    overall_feedback = f'Overall feedback: {state["language_feedback"]} {state["analysis_feedback"]} {state["clarity_feedback"]}'
    return {'avg_score': avg_score, 'overall_feedback': overall_feedback}

In [56]:
graph = StateGraph(UpscState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')

graph.add_edge('final_evaluation',END)

work_flow = graph.compile()


In [57]:
initial_state = {
    'essay': "My daily routine is very simple. I wake up in the morning and brush my teeth. After that I take breakfast and get ready for my work. I go to office and do my work. Sometimes work is very hard but I try my best. In evening I come back to home and take some rest. After dinner I use my phone and watch some videos. Then I go to sleep. This is my daily routine and I like it very much.",
}
work_flow.invoke(initial_state)

{'essay': 'My daily routine is very simple. I wake up in the morning and brush my teeth. After that I take breakfast and get ready for my work. I go to office and do my work. Sometimes work is very hard but I try my best. In evening I come back to home and take some rest. After dinner I use my phone and watch some videos. Then I go to sleep. This is my daily routine and I like it very much.',
 'language_feedback': "The essay presents a clear and straightforward outline of your daily routine, which is good for clarity. However, the language is quite basic and lacks variety. You could improve the quality by incorporating more descriptive language and varied sentence structures. For instance, instead of saying 'I wake up in the morning and brush my teeth', you could say 'Upon awakening, I immediately brush my teeth to start the day fresh.' Additionally, expanding on your experiences or feelings about different parts of your routine could make the essay more engaging. The conclusion could 